In [ ]:
# %reset -f
%load_ext autoreload
%autoreload complete --log


In [ ]:
from smtgraphformer import *
from smtgraphformer.smtGraphFormer import SMTConfig
from trainModel import parseConfig

setDisplayOptions()
sr = setReproducibility(17711)


### Setup Search Space


In [ ]:
fp_baseline = Path("../models/ablation/smtM24-06042137/trainModel.yaml")
assert fp_baseline.exists(), "!!!"

cfg_baseline = yamlLoader(fp_baseline)
cfg_baseline["model"]["model_dir"] = "../models/sensitivity"

fp_configs = Path("../configs/sensitivity")
fp_configs.mkdir(parents=True, exist_ok=True)

searchSpace = {
    "learning_rate": {"kind": "log", "low": 1.5e-4, "high": 6.0e-4},
    "w_decay": {"kind": "log", "low": 5.0e-4, "high": 2.0e-2},
    "dropout_pct": {"kind": "float", "low": 0.0, "high": 0.25},
    "pct_warmup": {"kind": "float", "low": 0.02, "high": 0.15},
    "w_surrogate_tasks": {"kind": "choice", "values": [0.25, 0.5, 0.75, 1.0, 1.25]},
    "max_grad_norm": {"kind": "choice", "values": [0.5, 1.0, 2.0, 5.0]},
}


### Generate Plan & Model Configs


In [ ]:
def latinAxis(n_runs, seed=17711):
    rng = np.random.default_rng(seed)
    axis = (np.arange(n_runs) + rng.random(n_runs)) / n_runs
    return axis[rng.permutation(n_runs)]


def scaleValue(u, spec):
    kind = spec["kind"]
    if kind == "log":
        low = np.log10(spec["low"])
        high = np.log10(spec["high"])
        return float(10 ** (low + u * (high - low)))
    if kind == "float":
        return float(spec["low"] + u * (spec["high"] - spec["low"]))
    if kind == "choice":
        values = spec["values"]
        idx = min(int(u * len(values)), len(values) - 1)
        return values[idx]
    raise ValueError(kind)


def tidyValue(param, value):
    if param in {"dropout_pct", "pct_warmup"}:
        return float(f"{value:.3f}")
    if param in {"learning_rate", "w_decay"}:
        return float(f"{value:.6g}")
    return value


In [ ]:
def createSensitivityPlan(cfg_baseline, searchSpace, n_runs=20):
    l_plan = []
    lfcp = lambda tag: f"{fp_configs}/trainModel.{tag}.yaml"

    # include baseline for reproducibility and sanity checks
    r_baseline = {"$idx": 0, "$tag": "SA00"}
    for param in searchSpace:
        r_baseline[param] = cfg_baseline["model"][param]
    r_baseline["$path"] = lfcp("SA00")  # path for baseline config
    l_plan.append(r_baseline)

    # pre-generate Latin axes for each parameter
    sampled_axes = {
        param: latinAxis(n_runs, seed=17711 + offset)
        for offset, param in enumerate(searchSpace, start=1)
    }

    # include sampled configs
    for idx in range(1, n_runs + 1):
        row = {"$idx": idx, "$tag": f"SA{idx:02d}"}
        for param, spec in searchSpace.items():
            u = sampled_axes[param][idx - 1]  # idx-1 because baseline is at index 0
            row[param] = tidyValue(param, scaleValue(u, spec))

        row["$path"] = lfcp(f"SA{idx:02d}")  # path for this config
        l_plan.append(row)

    plan = pd.DataFrame(l_plan)
    df_csver(plan, f"{fp_configs}/SensitivityAnalysisPlan")
    print(f"{plan.shape=}")

    return plan


In [ ]:
def slicer(d: SMTConfig) -> dict:
    return {k.replace("use_", ""): v for k, v in vars(d).items() if k in searchSpace}


def nativeValue(value):
    return value.item() if hasattr(value, "item") else value


def createSensitivityConfig(idx, cfg_baseline: dict, searchSpace: dict):
    plan = pd.read_csv(f"{fp_configs}/SensitivityAnalysisPlan.csv")
    row = plan.loc[plan["$idx"] == idx].squeeze()
    fp_config = str(row["$path"])

    cfg = deepcopy(cfg_baseline)
    for param in searchSpace:
        cfg["model"][param] = nativeValue(row[param])

    with open(fp_config, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    # sanity check that config can be parsed; return path to config for training
    print(f"{row['$tag']}:", slicer(parseConfig(fp_config)[0]))
    return fp_config


### Run Sweep


In [ ]:
plan = createSensitivityPlan(cfg_baseline, searchSpace, n_runs=99)
# fp = createSensitivityConfig(17, cfg_baseline, searchSpace)  # sanity check
plan.head()


In [ ]:
# customise the sweep indices to run; use 0--100 for the full sweep including baseline
s_start, s_stop = 0, 100

with pipeCellOutput(f"{fp_configs}/sensitivitySweep.log"):
    for idx in range(s_start, s_stop):
        print(f"\n--- Sensitivity: {idx} ---")
        fp_config = createSensitivityConfig(idx, cfg_baseline, searchSpace)
        !python trainModel.py -c {fp_config}

        # break


In [ ]:
results = summariseFolderResults("../models/")
print(results.head())


### end